# Persian Tweet Text Generation — LSTM 


## 0. Setup

In [ ]:
import subprocess, sys, time

# Install system package for 7z extraction
print('Installing p7zip...')
subprocess.run(['apt-get', 'install', '-y', 'p7zip-full'], capture_output=True)

# Install Python packages
print('Installing Python packages...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'tqdm', 'hazm'], capture_output=True)
print('All installations complete.')

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
import os, math, time, re, json, collections
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from tqdm import tqdm
from hazm import Normalizer

# ── Paths ────────────────────────────────────────────────────────────────
DRIVE_PATH   = '/content/drive/MyDrive/twitter_sample_tweets.csv.7z'
DATA_DIR     = '/content/data'
CLEANED_CSV  = '/content/cleaned_tweets.csv'
DATA_PATH    = CLEANED_CSV
CKPT_DIR     = '/content/drive/MyDrive/persian_lstm_checkpoints'
MODEL_DIR    = '/content/drive/MyDrive/persian_lstm_checkpoints/final'

# ── Data ─────────────────────────────────────────────────────────────────
N_TWEETS     = 80_000          MAX_LENGTH   = 32              
# ── Model architecture ───────────────────────────────────────────────────
EMBED_SIZE   = 128     # token embedding size
HIDDEN_SIZE  = 256     # LSTM hidden state size
NUM_LAYERS   = 2       # stacked LSTM layers
DROPOUT      = 0.2     # applied between LSTM layers and before FC

# ── Training ─────────────────────────────────────────────────────────────
BATCH_SIZE   = 32
EPOCHS       = 10
LR           = 1e-3
WARMUP_RATIO = 0.1    # 10 % of total stepsLABEL_SMOOTH = 0.1
GRAD_CLIP    = 1.0

import random
random.seed(42)
torch.manual_seed(42)

## 1. Mount Google Drive & Extract / Preprocess Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
print(f'Checkpoint dir : {CKPT_DIR}')
print(f'Final model dir: {MODEL_DIR}')
extracted_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.csv')]
if extracted_files:
    RAW_CSV = os.path.join(DATA_DIR, extracted_files[0])
    print(f'Already extracted: {RAW_CSV}')
else:
    print(f'Extracting {DRIVE_PATH} ...')
    t0 = time.time()
    result = subprocess.run(
        ['7z', 'x', DRIVE_PATH, f'-o{DATA_DIR}', '-y'],
        capture_output=True, text=True
    )
    print(result.stdout[-500:] if result.stdout else 'No output')
    if result.returncode != 0:
        print('STDERR:', result.stderr[-300:])
    print(f'Done in {(time.time()-t0)/60:.1f} min')
    extracted_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.csv')]
    assert extracted_files, 'No CSV found after extraction!'
    RAW_CSV = os.path.join(DATA_DIR, extracted_files[0])
    print(f'CSV: {RAW_CSV}  ({os.path.getsize(RAW_CSV)/1e9:.2f} GB)')

In [ ]:
PERSIAN_RE = re.compile(r'[\u0600-\u06FF]')

def persian_ratio(s):
    s = str(s)
    if not s:
        return 0.0
    return len(PERSIAN_RE.findall(s)) / len(s)

def detect_header_and_text_column(csv_path, n_peek=10):
    peek = pd.read_csv(csv_path, header=None, nrows=n_peek, on_bad_lines='skip',
                       encoding='utf-8', low_memory=False)
    print(f'Peek shape: {peek.shape}')
    print(peek.head(3).to_string())
    row0_ratios = [persian_ratio(peek.iloc[0, c]) for c in range(peek.shape[1])]
    data_ratios = []
    for c in range(peek.shape[1]):
        col_data = peek.iloc[1:, c].astype(str)
        data_ratios.append(col_data.apply(persian_ratio).mean())
    avg_row0 = np.mean(row0_ratios)
    avg_data = np.mean(data_ratios)
    has_header = avg_row0 < 0.1 and avg_data > 0.05
    print(f'Row-0 avg Persian ratio: {avg_row0:.3f}')
    print(f'Data rows avg Persian ratio: {avg_data:.3f}')
    print(f'Detected header row: {has_header}')
    peek_data = peek.iloc[1:] if has_header else peek
    scores = []
    for c in range(peek_data.shape[1]):
        col = peek_data.iloc[:, c].astype(str)
        pr = col.apply(persian_ratio).mean()
        ln = col.apply(len).mean()
        scores.append((pr, ln))
    max_ln = max(s[1] for s in scores) or 1
    combined = [0.5*s[0] + 0.5*(s[1]/max_ln) for s in scores]
    text_col_idx = int(np.argmax(combined))
    print(f'Selected text column index: {text_col_idx}')
    return has_header, text_col_idx

HAS_HEADER, TEXT_COL_IDX = detect_header_and_text_column(RAW_CSV)

In [ ]:
normalizer = Normalizer()

URL_RE     = re.compile(r'https?://\S+')
MENTION_RE = re.compile(r'@\w+')
HASHTAG_RE = re.compile(r'#')
DIGIT_RE   = re.compile(r'[0-9\u06F0-\u06F9]')
KEEP_RE    = re.compile(r'[^\u0600-\u06FF .\u060C,!?\s]')
REPEAT_RE  = re.compile(r'(.)\1{2,}')
SPACE_RE   = re.compile(r'\s+')

def clean_tweet(text):
    text = str(text)
    text = URL_RE.sub('', text)
    text = MENTION_RE.sub('', text)
    text = HASHTAG_RE.sub('', text)
    text = DIGIT_RE.sub('', text)
    text = KEEP_RE.sub('', text)
    try:
        text = normalizer.normalize(text)
    except Exception:
        pass
    text = REPEAT_RE.sub(r'\1\1', text)
    text = SPACE_RE.sub(' ', text).strip()
    return text

def word_count(text):
    return len(text.split())

sample = 'امروز به https://example.com دانشگاه رفتم @user123 #خوب'
print('Before:', sample)
print('After :', clean_tweet(sample))

In [ ]:
if os.path.exists(CLEANED_CSV) and os.path.getsize(CLEANED_CSV) > 1_000_000:
    print(f'Cleaned CSV already exists: {CLEANED_CSV}')
else:
    print('Preprocessing tweets...')
    CHUNK_SIZE = 100_000
    seen = set()
    total_raw = 0
    total_written = 0
    csv_kwargs = dict(
        chunksize=CHUNK_SIZE,
        usecols=[TEXT_COL_IDX],
        header=0 if HAS_HEADER else None,
        on_bad_lines='skip',
        encoding='utf-8',
        low_memory=False,
    )
    t0 = time.time()
    with open(CLEANED_CSV, 'w', encoding='utf-8') as fout:
        fout.write('text\n')  # header
        reader = pd.read_csv(RAW_CSV, **csv_kwargs)
        for chunk_idx, chunk in enumerate(reader):
            if total_written >= N_TWEETS:
                break
            chunk.columns = ['text']
            chunk = chunk.dropna(subset=['text'])
            chunk['cleaned'] = chunk['text'].apply(clean_tweet)
            chunk = chunk[chunk['cleaned'].str.len() > 0]
            chunk = chunk[chunk['cleaned'].apply(word_count) >= 5]
            chunk = chunk[~chunk['cleaned'].isin(seen)]
            seen.update(chunk['cleaned'].tolist())
            remaining = N_TWEETS - total_written
            chunk = chunk.head(remaining)
            total_raw += CHUNK_SIZE
            total_written += len(chunk)
            for line in chunk['cleaned']:
                fout.write(line + '\n')
            elapsed = time.time() - t0
            print(f'Chunk {chunk_idx+1:3d} | raw~{total_raw:,} | kept={total_written:,} | '
                  f'elapsed={elapsed/60:.1f}min', end='\r')
    print(f'\nDone. Lines: {total_written:,}  File: {os.path.getsize(CLEANED_CSV)/1e6:.1f} MB')

## 2. Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('HooshvareLab/gpt2-fa')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

VOCAB_SIZE = len(tokenizer)
PAD_ID     = tokenizer.pad_token_id
EOS_ID     = tokenizer.eos_token_id

print(f'Vocab size : {VOCAB_SIZE:,}')
print(f'PAD token  : {tokenizer.pad_token!r}  (id={PAD_ID})')
print(f'EOS token  : {tokenizer.eos_token!r}  (id={EOS_ID})')

## 3. Load & Tokenize Data

In [ ]:
df     = pd.read_csv(DATA_PATH, usecols=['text'], nrows=N_TWEETS)
tweets = df['text'].astype(str).dropna().tolist()
tweets = [re.sub(r'_', ' ', t) for t in tweets]
tweets = [re.sub(r'\s+', ' ', t).strip() for t in tweets]
tweets = [t for t in tweets if len(t) > 5]

print(f'Tweets loaded  : {len(tweets):,}')
print(f'Example        : {tweets[1]}')

print('Tokenizing...')
t0 = time.time()
encodings = tokenizer(
    tweets,
    add_special_tokens=True,
    padding='max_length',
    truncation=True,
    max_length=MAX_LENGTH,
)
print(f'Done in {time.time()-t0:.1f}s')

In [ ]:
class TweetDataset(Dataset):
    def __init__(self, encodings):
        self.input_ids = encodings['input_ids']

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        ids = self.input_ids[idx]
        x   = torch.tensor(ids[:-1], dtype=torch.long)  # input tokens
        y   = torch.tensor(ids[1:],  dtype=torch.long)  # shifted targets
        return x, y

dataset    = TweetDataset(encodings)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                        pin_memory=(device.type == 'cuda'), num_workers=2)
print(f'Dataset size   : {len(dataset):,}')
print(f'Batches/epoch  : {len(dataloader):,}')

## 4. LSTM Model


In [ ]:
class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers, dropout):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_size, padding_idx=PAD_ID)
        self.lstm  = nn.LSTM(
            embed_size, hidden_size, num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
        )
        self.dropout = nn.Dropout(dropout)
        self.ln      = nn.LayerNorm(hidden_size)   # LayerNorm stabilises trainingself.fc      = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden=None):
        emb         = self.dropout(self.embed(x))
        out, hidden = self.lstm(emb, hidden)
        out         = self.ln(out)
        logits      = self.fc(self.dropout(out))
        return logits, hidden

model    = LSTMLanguageModel(VOCAB_SIZE, EMBED_SIZE, HIDDEN_SIZE, NUM_LAYERS, DROPOUT).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'LSTM parameters: {n_params:,}')

## 5. Training

- After each epoch, a checkpoint is saved to Drive
- The previous checkpoint is deleted to save space
- If the session disconnects, just re-run this cell — training resumes from the last epoch

In [ ]:
# ignore PAD positions; label smoothing reduces overconfidence
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=LABEL_SMOOTH)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

# linear warmup then cosine annealingtotal_steps  = EPOCHS * len(dataloader)
warmup_steps = int(total_steps * WARMUP_RATIO)

# LR schedule: linear warmup then cosine annealing
def lr_lambda(step):
    if step < warmup_steps:
        return float(step) / max(1, warmup_steps)
    progress = float(step - warmup_steps) / max(1, total_steps - warmup_steps)
    return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# ── Checkpoint helpers ────────────────────────────────────────────────────
def ckpt_path(epoch):
    return os.path.join(CKPT_DIR, f'lstm_epoch{epoch:02d}.pt')

def find_latest_ckpt():
    found = []
    for fname in os.listdir(CKPT_DIR):
        if fname.startswith('lstm_epoch') and fname.endswith('.pt'):
            try:
                ep = int(fname.replace('lstm_epoch','').replace('.pt',''))
                found.append((ep, os.path.join(CKPT_DIR, fname)))
            except ValueError:
                pass
    return sorted(found)[-1] if found else (0, None)

def save_ckpt(epoch, stats, prev_path):
    path = ckpt_path(epoch)
    torch.save({
        'epoch':           epoch,
        'model_state':     model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'losses':          stats['losses'],
        'perplexities':    stats['perplexities'],
        'epoch_times':     stats['epoch_times'],
    }, path)
    print(f'  Checkpoint saved -> {path}')
    # delete previous checkpoint to save Drive spaceif prev_path and os.path.exists(prev_path):
        os.remove(prev_path)
        print(f'  Deleted previous -> {prev_path}')
    return path

def load_ckpt():
    last_ep, path = find_latest_ckpt()
    if path is None:
        print('No checkpoint found — starting from epoch 1.')
        return 0, {'losses': [], 'perplexities': [], 'epoch_times': []}, None
    print(f'Resuming from epoch {last_ep} -> {path}')
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    stats = {
        'losses':       ckpt['losses'],
        'perplexities': ckpt['perplexities'],
        'epoch_times':  ckpt['epoch_times'],
    }
    return last_ep, stats, path

# ── Training loop ─────────────────────────────────────────────────────────
start_ep, stats, last_ckpt = load_ckpt()

if start_ep >= EPOCHS:
    print(f'Already fully trained ({EPOCHS} epochs). Skipping.')
else:
    print(f'\n' + '='*60)
    print(f' Training LSTM  (epochs {start_ep+1}-{EPOCHS})')
    print(f' warmup={warmup_steps} steps | total={total_steps} steps')
    print('='*60)

    for epoch in range(start_ep + 1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        t0         = time.time()

        loop = tqdm(dataloader, desc=f'Epoch {epoch}/{EPOCHS}', leave=True)
        for batch_idx, (x, y) in enumerate(loop, start=1):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits, _ = model(x)
            loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()
            avg = total_loss / batch_idx
            loop.set_postfix(loss=f'{avg:.4f}', ppl=f'{math.exp(min(avg,20)):.1f}',
                             lr=f'{scheduler.get_last_lr()[0]:.2e}')

        elapsed  = time.time() - t0
        avg_loss = total_loss / len(dataloader)
        ppl      = math.exp(min(avg_loss, 20))
        stats['losses'].append(avg_loss)
        stats['perplexities'].append(ppl)
        stats['epoch_times'].append(elapsed / 60)

        print(f'\n=== Epoch {epoch} Summary ===')
        print(f'  Loss       : {avg_loss:.4f}')
        print(f'  Perplexity : {ppl:.2f}')
        print(f'  Time       : {elapsed/60:.1f} min')

        last_ckpt = save_ckpt(epoch, stats, last_ckpt)

    print(f'\nTotal training time: {sum(stats["epoch_times"]):.1f} min')

## 6. Training Curves

In [ ]:
epochs_x = list(range(1, len(stats['losses']) + 1))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs_x, stats['losses'], 'o-', color='steelblue')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-Entropy')
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_x, stats['perplexities'], 'o-', color='tomato')
axes[1].set_title('Perplexity')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('PPL')
axes[1].grid(alpha=0.3)

axes[2].bar(epochs_x, stats['epoch_times'], color='steelblue', alpha=0.8)
axes[2].set_title('Time per Epoch (min)')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Minutes')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/lstm_training.png', dpi=120)
plt.show()

## 7. Save final model

In [ ]:
torch.save(model.state_dict(), os.path.join(MODEL_DIR, 'model.pt'))
tokenizer.save_pretrained(os.path.join(MODEL_DIR, 'tokenizer'))
print(f'Model saved    -> {MODEL_DIR}/model.pt')
print(f'Tokenizer saved-> {MODEL_DIR}/tokenizer/')

## 8. Text Generation


In [ ]:
def generate(seed_text='', max_new_tokens=30,
             temperature=0.8, top_p=0.9, rep_penalty=1.3):
    model.eval()
    ids    = tokenizer.encode(seed_text, add_special_tokens=False)
    if not ids:
        ids = [EOS_ID]
    hidden = None

    # warm up hidden state on the prompt tokensinput_ids = torch.tensor([ids], dtype=torch.long, device=device)
    with torch.no_grad():
        _, hidden = model(input_ids, hidden)

    generated = []   # newly generated token IDs (excluding the prompt)
    last_id   = torch.tensor([[ids[-1]]], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits, hidden = model(last_id, hidden)
        next_logits = logits[0, -1] / temperature

        # penalise tokens already in the sequence to reduce repetitionfor tok_id in set(ids + generated):
            if next_logits[tok_id] > 0:
                next_logits[tok_id] /= rep_penalty
            else:
                next_logits[tok_id] *= rep_penalty

        # nucleus (top-p) sampling: keep the smallest set of tokens whose cumulative prob ≥ top_psorted_logits, sorted_idx = torch.sort(next_logits, descending=True)
        cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
        sorted_logits[cum_probs - F.softmax(sorted_logits, dim=-1) > top_p] = float('-inf')
        next_logits = torch.zeros_like(next_logits).scatter_(0, sorted_idx, sorted_logits)
        probs   = F.softmax(next_logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1).item()

        if next_id == EOS_ID:
            break
        generated.append(next_id)
        last_id = torch.tensor([[next_id]], dtype=torch.long, device=device)

    return tokenizer.decode(ids + generated, skip_special_tokens=True)

PROMPTS = [
    'امروز',
    'دانشگاه',
    'ایران زیبا',
]

print('\n' + '='*60)
print('GENERATION RESULTS')
print('='*60)
for prompt in PROMPTS:
    result = generate(prompt)
    print(f'\n  Prompt : {prompt}')
    print(f'  Output : {result}')

## 9. Summary

In [ ]:
print('='*55)
print('LSTM SUMMARY')
print('='*55)
print(f'Tokenizer   : HooshvareLab/gpt2-fa  (vocab={VOCAB_SIZE:,})')
print(f'Architecture: LSTM  embed={EMBED_SIZE}  hidden={HIDDEN_SIZE}  layers={NUM_LAYERS}')
print(f'Parameters  : {n_params:,}')
print(f'Seq length  : {MAX_LENGTH}')
print(f'Batch size  : {BATCH_SIZE}')
print(f'Epochs      : {len(stats["losses"])}/{EPOCHS}')
if stats['losses']:
    print(f'Final loss  : {stats["losses"][-1]:.4f}')
    print(f'Final PPL   : {stats["perplexities"][-1]:.2f}')
    print(f'Total time  : {sum(stats["epoch_times"]):.1f} min')
print()
print('Files on Drive:')
for p in [last_ckpt, os.path.join(MODEL_DIR, 'model.pt')]:
    if p and os.path.exists(p):
        size = os.path.getsize(p)/1e6
        print(f'  OK   {p}  ({size:.1f} MB)')
print('\nDone!')